# Agent-Diff Benchmark: ReAct Agent

Run the [Agent-Diff benchmark](https://arxiv.org/abs/2602.11224) using the paper's custom ReAct agent loop with XML-tag parsing.

The agent reasons step-by-step (`<thinking>`), executes bash/curl commands (`<action>`), observes the result, and repeats until the task is done (`<done>`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agent-diff-bench/agent-diff/blob/main/examples/react_agent_benchmark.ipynb)

**Links:** [Paper](https://arxiv.org/abs/2602.11224) | [Dataset](https://huggingface.co/datasets/hubertmarek/agent-diff-bench) | [GitHub](https://github.com/agent-diff-bench/agent-diff)

In [ ]:
# Dependencies are managed by the active uv environment.
# From the repository root: uv pip install -e sdk/agent-diff-python httpx datasets jupyterlab

In [ ]:
# Configure these in the shell before starting Jupyter:
# export AGENT_DIFF_BASE_URL=http://127.0.0.1:8000
# export OPENAI_BASE_URL=http://127.0.0.1:8001/v1
# export OPENAI_MODEL=<served-model-name>
# export OPENAI_API_KEY=<optional-local-api-key>
# export REACT_TRACE_DIR=examples/evaluation_outputs/react_agent/checkpoints
# export REACT_CHECKPOINT_PATH=<checkpoint-to-create-or-resume>

In [ ]:
import os
import re
import time
import json
import httpx
import random
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_dataset
from agent_diff import AgentDiff, BashExecutorProxy

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:8001/v1").rstrip("/")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = os.getenv("OPENAI_MODEL")
if not MODEL:
    raise ValueError("Set OPENAI_MODEL to the model name exposed by your local server")

SERVICE_CONFIG = {
    "slack": {"name": "Slack", "base_url": "https://slack.com/api", "extra": ""},
    "box": {"name": "Box", "base_url": "https://api.box.com/2.0", "extra": ""},
    "calendar": {"name": "Google Calendar", "base_url": "https://www.googleapis.com/calendar/v3",
                 "extra": "- **Current Date/Time**: Sunday, June 17, 2018 at 00:01 (midnight), timezone America/Los_Angeles.\n"},
    "linear": {"name": "Linear", "base_url": "https://api.linear.app/graphql", "extra": ""},
}

REACT_SYSTEM_PROMPT = """You are an AI assistant that completes tasks by interacting with APIs via bash commands.

## Current Session
- **Service**: {service_name}
- **Base URL**: {base_url}
{extra_context}

## Environment
- Authentication is handled automatically via proxy. Use placeholder tokens where credentials would go.
- You execute bash commands (primarily curl) to interact with the API.
- If you are not sure how to use the API, explore the endpoint, parameters, and learn how it works.

## Response Format
Respond using XML tags:

<thinking>Your reasoning</thinking>
<action>Your bash command</action>

When done:
<thinking>Your reasoning</thinking>
<done>Brief summary</done>

## Rules
1. Execute ONE command at a time, then wait for the result.
2. Parse API responses carefully - extract IDs and data needed for subsequent calls.
3. If a command fails, analyze the error and try a different approach.
4. Only use <done> when the task is fully completed.
"""


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def normalize_usage(usage):
    usage = usage or {}
    return {
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_tokens": usage.get("total_tokens", 0),
        "cost": usage.get("cost", 0.0),
    }


def call_openai_compatible(model, messages, max_retries=3, on_attempt=None):
    headers = {}
    if OPENAI_API_KEY:
        headers["Authorization"] = f"Bearer {OPENAI_API_KEY}"

    attempts = []
    for attempt_index in range(max_retries):
        started = time.perf_counter()
        try:
            with httpx.Client(timeout=120) as http:
                resp = http.post(
                    f"{OPENAI_BASE_URL}/chat/completions",
                    headers=headers,
                    json={"model": model, "messages": messages},
                )
                resp.raise_for_status()
                data = resp.json()
                choice = data["choices"][0]
                attempt = {
                    "attempt": attempt_index + 1,
                    "status": "success",
                    "http_status": resp.status_code,
                    "elapsed_seconds": round(time.perf_counter() - started, 3),
                }
                attempts.append(attempt)
                if on_attempt:
                    on_attempt(attempt)
                return {
                    "content": choice["message"].get("content") or "",
                    "finish_reason": choice.get("finish_reason"),
                    "usage": normalize_usage(data.get("usage")),
                    "raw_response": data,
                    "attempts": attempts,
                }
        except (httpx.HTTPStatusError, httpx.RequestError, ValueError, KeyError) as exc:
            attempt = {
                "attempt": attempt_index + 1,
                "status": "error",
                "error_type": type(exc).__name__,
                "error": str(exc),
                "elapsed_seconds": round(time.perf_counter() - started, 3),
            }
            if isinstance(exc, httpx.HTTPStatusError):
                attempt["http_status"] = exc.response.status_code
                attempt["response_body"] = exc.response.text
            attempts.append(attempt)
            if on_attempt:
                on_attempt(attempt)
            if attempt_index < max_retries - 1:
                time.sleep(2 * (2 ** attempt_index) + random.uniform(0, 1))
                continue
            raise


def parse_react(response):
    thinking = re.search(r'<thinking>(.*?)</thinking>', response, re.DOTALL)
    action = re.search(r'<action>(.*?)</action>', response, re.DOTALL)
    done = re.search(r'<done>(.*?)</done>', response, re.DOTALL)
    return (
        thinking.group(1).strip() if thinking else None,
        action.group(1).strip() if action else None,
        done.group(1).strip() if done else None,
    )


def run_react_agent(model, prompt, bash, system_prompt, max_iterations=40, on_trace_update=None):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    trace = {
        "steps": [],
        "final": None,
        "iterations": 0,
        "completed": False,
        "usage": normalize_usage({}),
        "messages": messages,
    }

    def emit(stage):
        trace["stage"] = stage
        trace["updated_at"] = utc_now()
        if on_trace_update:
            on_trace_update(trace, stage)

    def add_usage(iter_usage):
        for key in ("prompt_tokens", "completion_tokens", "total_tokens", "cost"):
            trace["usage"][key] += iter_usage.get(key, 0)

    emit("agent_started")
    for i in range(max_iterations):
        step = {
            "iteration": i + 1,
            "status": "requesting_model",
            "request": {"model": model, "message_count": len(messages)},
            "api_attempts": [],
        }
        trace["steps"].append(step)
        trace["iterations"] = i + 1
        emit("model_request_started")

        def record_attempt(attempt):
            step["api_attempts"].append(attempt)
            emit("model_attempt_finished")

        try:
            api_response = call_openai_compatible(model, messages, on_attempt=record_attempt)
        except Exception as exc:
            step.update({
                "status": "model_error",
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            trace["error"] = {"type": type(exc).__name__, "message": str(exc)}
            emit("model_error")
            break

        text = api_response["content"]
        thinking, action, done = parse_react(text)
        iter_usage = api_response["usage"]
        add_usage(iter_usage)
        step.update({
            "status": "model_response_received",
            "thinking": thinking,
            "action": action,
            "done": done,
            "raw_response": text,
            "api_response": api_response["raw_response"],
            "finish_reason": api_response["finish_reason"],
            "usage": iter_usage,
        })
        messages.append({"role": "assistant", "content": text})
        emit("model_response_received")

        if action:
            try:
                executor_result = bash.execute(action)
                if isinstance(executor_result, dict):
                    observation = {
                        "stdout": executor_result.get("stdout", ""),
                        "stderr": executor_result.get("stderr", ""),
                        "exit_code": executor_result.get("exit_code", 0),
                    }
                else:
                    observation = {"stdout": str(executor_result or ""), "stderr": "", "exit_code": 0}
            except Exception as exc:
                executor_result = {"error": str(exc)}
                observation = {
                    "stdout": "",
                    "stderr": str(exc),
                    "exit_code": 1,
                    "error_type": type(exc).__name__,
                }
            step["executor_result"] = executor_result
            step["observation"] = observation
            step["status"] = "observation_recorded"
            obs = observation["stdout"].strip() or "(empty output)"
            if observation["exit_code"] != 0:
                obs = f"{observation['stdout']}\n[stderr]: {observation['stderr']}\n[exit_code]: {observation['exit_code']}".strip()
            messages.append({"role": "user", "content": f"<observation>\n{obs}\n</observation>"})
            emit("observation_recorded")
        elif done:
            trace["final"] = {
                "iteration": i + 1,
                "thinking": thinking,
                "summary": done,
                "raw_response": text,
                "api_response": api_response["raw_response"],
                "finish_reason": api_response["finish_reason"],
                "usage": iter_usage,
            }
            trace["steps"].pop()
            trace["completed"] = True
            emit("agent_completed")
            break
        else:
            step["status"] = "protocol_warning"
            step["warning"] = "No <action> or <done> tag found"
            messages.append({"role": "user", "content": "Please respond with <action> or <done>."})
            emit("protocol_warning")

    if not trace["completed"] and "error" not in trace and trace["iterations"] >= max_iterations:
        trace["error"] = {"type": "MaxIterationsExceeded", "message": f"Reached {max_iterations} iterations"}
        emit("max_iterations_reached")
    return trace

In [ ]:
TRACE_SCHEMA_VERSION = 1
DATASET_NAME = "hubertmarek/agent-diff-bench"
DATASET_SPLIT = "train+test"
MAX_ITERATIONS = 40


def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "examples" / "react_agent_benchmark.ipynb").is_file():
            return candidate
    raise RuntimeError("Could not locate the agent-diff repository root")


REPO_ROOT = find_repo_root()
DEFAULT_TRACE_DIR = REPO_ROOT / "examples" / "evaluation_outputs" / "react_agent" / "checkpoints"


def json_default(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    return str(value)


def atomic_write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    try:
        with temp_path.open("w", encoding="utf-8") as handle:
            json.dump(data, handle, ensure_ascii=False, indent=2, default=json_default)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def resolve_checkpoint_path():
    override = os.getenv("REACT_CHECKPOINT_PATH")
    if override:
        path = Path(override).expanduser()
        return path if path.is_absolute() else REPO_ROOT / path
    trace_dir = Path(os.getenv("REACT_TRACE_DIR", str(DEFAULT_TRACE_DIR))).expanduser()
    if not trace_dir.is_absolute():
        trace_dir = REPO_ROOT / trace_dir
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return trace_dir / f"checkpoint_{timestamp}.json"


class ReactCheckpoint:
    COMPATIBILITY_FIELDS = (
        "model",
        "openai_base_url",
        "agent_diff_base_url",
        "dataset_name",
        "dataset_split",
        "dataset_fingerprint",
        "max_iterations",
    )

    def __init__(self, path, metadata):
        self.path = Path(path)
        if self.path.exists():
            with self.path.open(encoding="utf-8") as handle:
                self.data = json.load(handle)
            self._validate(metadata)
            self.data.setdefault("completed", [])
            self.data.setdefault("results", [])
            self.data.setdefault("in_progress", {})
            selected = set(self.data["metadata"].get("selected_test_ids", []))
            selected.update(metadata.get("selected_test_ids", []))
            self.data["metadata"]["selected_test_ids"] = sorted(selected)
            self.data["metadata"]["resumed_at"] = utc_now()
            print(f"[CHECKPOINT] Resuming {self.path}")
        else:
            self.data = {
                "metadata": metadata,
                "completed": [],
                "in_progress": {},
                "results": [],
                "saved_at": utc_now(),
            }
            print(f"[CHECKPOINT] Creating {self.path}")
        self.save()

    def _validate(self, metadata):
        existing = self.data.get("metadata", {})
        if existing.get("schema_version") != TRACE_SCHEMA_VERSION:
            raise ValueError("Checkpoint schema is not supported for resume")
        mismatches = []
        for field in self.COMPATIBILITY_FIELDS:
            if existing.get(field) != metadata.get(field):
                mismatches.append(f"{field}: {existing.get(field)!r} != {metadata.get(field)!r}")
        if mismatches:
            raise ValueError("Checkpoint configuration mismatch: " + "; ".join(mismatches))

    @staticmethod
    def task_key(model, test_id, run_index=0):
        return f"{model}|{test_id}|{run_index}"

    def is_completed(self, key):
        return key in set(self.data["completed"])

    def start_task(self, key, record):
        record["_checkpoint_key"] = key
        self.data["in_progress"][key] = record
        self.save()

    def update_task(self, key, record):
        record["_checkpoint_key"] = key
        self.data["in_progress"][key] = record
        self.save()

    def complete_task(self, key, record):
        record["_checkpoint_key"] = key
        self.data["in_progress"].pop(key, None)
        self.data["results"] = [
            item for item in self.data["results"] if item.get("_checkpoint_key") != key
        ]
        self.data["results"].append(record)
        self.data["completed"] = sorted(set(self.data["completed"]) | {key})
        self.save()

    def results_for(self, keys):
        keys = set(keys)
        return [item for item in self.data["results"] if item.get("_checkpoint_key") in keys]

    def save(self):
        self.data["saved_at"] = utc_now()
        atomic_write_json(self.path, self.data)

In [ ]:
client = AgentDiff()
dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
selected_dataset = dataset
selected_test_ids = [str(example["test_id"]) for example in selected_dataset]

checkpoint_metadata = {
    "schema_version": TRACE_SCHEMA_VERSION,
    "created_at": utc_now(),
    "model": MODEL,
    "openai_base_url": OPENAI_BASE_URL,
    "agent_diff_base_url": client.base_url,
    "dataset_name": DATASET_NAME,
    "dataset_split": DATASET_SPLIT,
    "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
    "max_iterations": MAX_ITERATIONS,
    "selected_test_ids": selected_test_ids,
}
checkpoint = ReactCheckpoint(resolve_checkpoint_path(), checkpoint_metadata)
selected_keys = [ReactCheckpoint.task_key(MODEL, test_id) for test_id in selected_test_ids]

for example in selected_dataset:
    test_id = str(example["test_id"])
    key = ReactCheckpoint.task_key(MODEL, test_id)
    test_name = example.get("test_name", test_id)
    if checkpoint.is_completed(key):
        print(f"[SKIP] {test_name} already completed")
        continue

    info = json.loads(example["info"]) if isinstance(example["info"], str) else example["info"]
    expected = json.loads(example["answer"]) if isinstance(example["answer"], str) else example["answer"]
    service = info["service"]
    cfg = SERVICE_CONFIG[service]
    system_prompt = REACT_SYSTEM_PROMPT.format(
        service_name=cfg["name"], base_url=cfg["base_url"], extra_context=cfg["extra"]
    )
    task_record = {
        "test_id": test_id,
        "test_name": test_name,
        "service": service,
        "model": MODEL,
        "run_index": 0,
        "prompt": example["question"],
        "system_prompt": system_prompt,
        "expected_output": expected,
        "status": "initializing",
        "passed": False,
        "score": None,
        "failures": [],
        "diff": None,
        "trace": None,
        "started_at": utc_now(),
    }
    checkpoint.start_task(key, task_record)
    print(f"Running: {test_name}")

    env = None
    run = None
    evaluation_completed = False
    start = time.perf_counter()
    try:
        env = client.init_env(
            templateService=info["service"],
            templateName=info["seed_template"],
            impersonateUserId=info["impersonate_user_id"],
        )
        task_record["environment_id"] = env.environmentId
        task_record["status"] = "environment_created"
        checkpoint.update_task(key, task_record)

        run = client.start_run(envId=env.environmentId)
        task_record["runId"] = run.runId
        task_record["status"] = "running"
        checkpoint.update_task(key, task_record)
        bash = BashExecutorProxy(env.environmentId, base_url=client.base_url, api_key=client.api_key)

        def persist_trace(trace, stage):
            task_record["trace"] = trace
            task_record["stage"] = stage
            task_record["status"] = "running"
            checkpoint.update_task(key, task_record)

        trace = run_react_agent(
            MODEL,
            example["question"],
            bash,
            system_prompt,
            max_iterations=MAX_ITERATIONS,
            on_trace_update=persist_trace,
        )
        task_record["trace"] = trace
        task_record["status"] = "evaluating"
        checkpoint.update_task(key, task_record)

        evaluation_response = client.evaluate_run(runId=run.runId, expectedOutput=expected)
        result = client.get_results_for_run(runId=run.runId)
        task_record.update({
            "status": result.status,
            "passed": result.passed,
            "score": result.score,
            "failures": result.failures,
            "diff": result.diff,
            "evaluation_response": evaluation_response,
            "evaluated_at": utc_now(),
        })
        evaluation_completed = True
    except Exception as exc:
        task_record["status"] = "error"
        task_record["error"] = {"type": type(exc).__name__, "message": str(exc)}
        checkpoint.update_task(key, task_record)
    finally:
        task_record["time"] = round(time.perf_counter() - start, 3)
        if env is not None:
            try:
                client.delete_env(envId=env.environmentId)
                task_record["environment_deleted"] = True
            except Exception as cleanup_exc:
                task_record["environment_deleted"] = False
                task_record["cleanup_error"] = {
                    "type": type(cleanup_exc).__name__,
                    "message": str(cleanup_exc),
                }
        task_record["finished_at"] = utc_now()
        if evaluation_completed:
            checkpoint.complete_task(key, task_record)
        else:
            checkpoint.update_task(key, task_record)

    if evaluation_completed:
        print(f"  {'PASS' if task_record['passed'] else 'FAIL'} | score={task_record['score']} | {task_record['time']:.1f}s")
    else:
        print(f"  ERROR | {task_record.get('error')} | {task_record['time']:.1f}s")

results = checkpoint.results_for(selected_keys)
passed = sum(1 for result in results if result.get("passed"))
print(f"\nResults: {passed}/{len(results)} passed; {len(results)}/{len(selected_keys)} completed")
print(f"Trace checkpoint: {checkpoint.path}")